## Get iCal files for Consumer Price Index release dates 

In [1]:
import re
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import requests

In [2]:
baseURL = "https://www.destatis.de/DE/Presse/Termine/2026/VPI_endgueltig_"

In [3]:
def get_vpi_ics(month):
    url = baseURL+month+".ics?view=renderICS"
    return requests.get(url)

In [4]:
response = get_vpi_ics("04")

In [5]:
print(response.text)


BEGIN:VCALENDAR
VERSION:2.0
PRODID:-//BUND//Government Site Builder//DE
BEGIN:VEVENT
UID:1433600
DTSTAMP:20260511T220000Z
DTSTART:20260511T220000Z
DTEND:20260511T220000Z
SUMMARY:Verbraucherpreisindex, April 2026 (Statisches Bundesamt)
END:VEVENT
END:VCALENDAR



In [6]:
def convert_ics_to_8am_berlin(ics: str) -> str:
    
    pattern = r"(\d{8}T\d{6})Z"

    def repl(match):
        dt_str = match.group(1)

        # Parse as UTC
        dt_utc = datetime.strptime(dt_str, "%Y%m%dT%H%M%S").replace(
            tzinfo=ZoneInfo("UTC")
        )

        # Convert to Berlin time
        dt_berlin = dt_utc.astimezone(ZoneInfo("Europe/Berlin"))

        # Replace time with 08:00 (same local date!)
        dt_berlin_8am = dt_berlin.replace(hour=8, minute=0, second=0)

        # Convert back to UTC
        dt_new_utc = dt_berlin_8am.astimezone(ZoneInfo("UTC"))

        # Return in original ICS format
        return dt_new_utc.strftime("%Y%m%dT%H%M%SZ")

    return re.sub(pattern, repl, ics)

In [7]:
result = convert_ics_to_8am_berlin(response.text)
print(result)


BEGIN:VCALENDAR
VERSION:2.0
PRODID:-//BUND//Government Site Builder//DE
BEGIN:VEVENT
UID:1433600
DTSTAMP:20260512T060000Z
DTSTART:20260512T060000Z
DTEND:20260512T060000Z
SUMMARY:Verbraucherpreisindex, April 2026 (Statisches Bundesamt)
END:VEVENT
END:VCALENDAR



In [8]:
for i in range(12):
    monthStr = (f"{i+1:02}")
    response = get_vpi_ics(monthStr)
    result = convert_ics_to_8am_berlin(response.text)
    with open("vpi_endgueltig_"+monthStr+".ics", "w") as f:
        f.write(result)